# Core chunk snapshot inspection

Inspection date: 2026-08-28. Corpus snapshot: `7077dab5f83cd024b87a30a1ab05a411804879b85a334a2e7b25f04562cdaa5f`. The private v2 build contains 14 datasets and 125,094 chunks; its metadata SHA-256 is `93cc8fd651a92b6581455a8e6b416ec4ea659cd235cf8c87f4aeb1f8b0ff15f0`. No source text or paid provider response is stored in this notebook.

In [ ]:
import json
from collections import Counter
from pathlib import Path

snapshot = "7077dab5f83cd024b87a30a1ab05a411804879b85a334a2e7b25f04562cdaa5f"
root = Path("../.ragbench/chunks") / f"{snapshot}-v2"
metadata = json.loads((root / "metadata.json").read_text(encoding="utf-8"))
assert len(metadata["datasets"]) == 14
mode_counts = Counter(item["mode"] for item in metadata["datasets"])
total_chunks = sum(item["chunks"] for item in metadata["datasets"])
mode_counts, total_chunks

## Samples and observations

- Text-heavy sample: `lge-sustainability-2024-2025`, source page 48. The PDF contains prose, headings, and a six-step human-rights process diagram. Fixed chunks preserve page provenance and overlap; heading chunks stay within the detected section. Upstage occasionally labels a full lead sentence as `heading1`, producing an overlong section label, but the text remains intact.
- Table-heavy sample: `kt-esg-appendix-2024`, source page 4. The GRI index headings now create distinct heading-aware groups. A table larger than 600 tokens is split token-wise, so some boundaries occur inside serialized HTML rows or tags. This is a known baseline limitation, not lost source content.
- The first build exposed that actual provider categories are `heading1` rather than `heading`; all section paths were empty and heading output duplicated fixed-600-100. The normalizer now recognizes `heading1` through `heading6`, and chunk snapshot IDs bind the exact derived JSONL bytes. In v2, heading and fixed share zero identical aligned records.

Decision: accept the seven required core strategies for retrieval screening. Keep row-aware table chunking as a later, separately versioned strategy only if table-question error analysis shows the baseline split is material.

In [ ]:
targets = {"lge-sustainability-2024-2025": 48, "kt-esg-appendix-2024": 4}
samples = []
for dataset in metadata["datasets"]:
    if dataset["strategy"] != "heading-600-100":
        continue
    with (root / dataset["path"]).open(encoding="utf-8") as source:
        for line in source:
            chunk = json.loads(line)
            page = targets.get(chunk["document_id"])
            if page and chunk["page_start"] <= page <= chunk["page_end"]:
                samples.append(
                    {
                        "mode": dataset["mode"],
                        "document_id": chunk["document_id"],
                        "page_range": (chunk["page_start"], chunk["page_end"]),
                        "tokens": chunk["token_count"],
                        "section_path": chunk["section_path"],
                    }
                )
samples